# Phase 1 — Depth Estimation
## Scale-Invariant 평가 + DepthAnythingV2 추론 + Detection 연동

### 학습 목표
- **Monocular Depth 기초** — 단안 카메라에서 깊이를 추정하는 원리와 한계
- **Scale-Invariant 평가 지표** — RMSE / AbsRel / δ1 직접 구현 (NYU Depth 표준)
- **DepthAnythingV2 아키텍처** — DINOv2 Encoder + DPT Decoder 원리 이해
- **Detection + Depth 연동** — YOLOv8 bbox → 객체별 거리 추정
- **자율주행 응용** — 거리 기반 위험도 분류 시스템

### Phase 1 전체 구조에서의 위치
```
Detection  → bbox(2D)        : 어디에 있는가?
Depth      → distance(m)     : 얼마나 멀리 있는가?
Seg + Depth→ 3D mask         : 정확히 어떤 공간을 차지하는가?
BEV (Ph3)  → 지면 좌표(X,Z)  : 지면에서 어디에 있는가?
```

## 0. 환경 확인

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import torch
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')
import transformers
print(f'Transformers: {transformers.__version__}')

PROJECT_ROOT = Path('C:/Users/apple/Desktop/autonomous_cv_pipeline')
COCO_IMG_DIR = Path.home() / 'MyProject' / 'datasets' / 'coco128' / 'images' / 'train2017'
COCO_LBL_DIR = Path.home() / 'MyProject' / 'datasets' / 'coco128' / 'labels' / 'train2017'
print(f'\nCOCO128 이미지: {len(list(COCO_IMG_DIR.glob("*.jpg")))}장')

## 1. Depth Estimation 기초 — 왜 어려운가?

### Monocular Depth의 근본 문제
```
3D → 2D 투영 (카메라):  정보 손실 발생
  (X, Y, Z) → (u, v)  ← Z(깊이) 정보 사라짐

2D → 3D 역투영 (우리가 하려는 것):
  (u, v) → (X, Y, Z)  ← Z를 모르면 무수히 많은 해 존재
```

### Scale Ambiguity 문제
단안 카메라는 **절대 거리**를 알 수 없음.
- 사람이 1m 앞에 있는지, 2m 앞에 2배 크기 사람이 있는지 구분 불가
- 모델이 출력하는 깊이는 **상대적 스케일** → 평가 시 정규화 필요

### 해결 전략
| 방법 | 원리 | 예시 |
|------|------|------|
| **데이터 기반** | 대규모 데이터로 스케일 학습 | DepthAnythingV2 |
| **기하학적** | 스테레오 카메라, LiDAR 보완 | 자율주행 차량 |
| **Scale-Invariant 평가** | 예측값을 GT 스케일로 정렬 후 평가 | 연구 표준 |

In [ ]:
# ══════════════════════════════════════════════════
# Depth 평가 지표 직접 구현
# 표준: NYU Depth v2, KITTI 벤치마크 기준
# ══════════════════════════════════════════════════

def align_scale_shift(pred: np.ndarray, gt: np.ndarray, mask: np.ndarray = None):
    """
    Scale-Invariant 정렬: 예측값을 GT 스케일로 맞춤.
    최소제곱법으로 pred * s + t = gt 의 s, t 계산.
    """
    if mask is None:
        mask = np.ones_like(gt, dtype=bool)
    p = pred[mask].flatten()
    g = gt[mask].flatten()
    # [s, t] = (A^T A)^{-1} A^T g
    A = np.stack([p, np.ones_like(p)], axis=1)
    result = np.linalg.lstsq(A, g, rcond=None)
    s, t = result[0]
    return pred * s + t, s, t


def compute_rmse(pred: np.ndarray, gt: np.ndarray, mask: np.ndarray = None) -> float:
    """Root Mean Squared Error (단위: 미터)"""
    if mask is None:
        mask = np.ones_like(gt, dtype=bool)
    diff = pred[mask] - gt[mask]
    return float(np.sqrt(np.mean(diff ** 2)))


def compute_abs_rel(pred: np.ndarray, gt: np.ndarray, mask: np.ndarray = None) -> float:
    """
    AbsRel (Absolute Relative Error).
    AbsRel = mean(|pred - gt| / gt)
    스케일에 무관한 상대 오차.
    """
    if mask is None:
        mask = np.ones_like(gt, dtype=bool)
    return float(np.mean(np.abs(pred[mask] - gt[mask]) / (gt[mask] + 1e-8)))


def compute_delta(pred: np.ndarray, gt: np.ndarray, threshold: float = 1.25,
                  mask: np.ndarray = None) -> float:
    """
    δ < threshold 비율 (Accuracy).
    δ1: threshold=1.25  (표준)
    δ2: threshold=1.25² 
    δ3: threshold=1.25³
    max(pred/gt, gt/pred) < threshold 인 픽셀 비율.
    """
    if mask is None:
        mask = np.ones_like(gt, dtype=bool)
    ratio = np.maximum(
        pred[mask] / (gt[mask] + 1e-8),
        gt[mask]   / (pred[mask] + 1e-8)
    )
    return float((ratio < threshold).mean())


def compute_si_rmse(pred: np.ndarray, gt: np.ndarray, mask: np.ndarray = None) -> float:
    """
    Scale-Invariant RMSE (Eigen et al. 2014).
    로그 공간에서 계산하여 스케일 무관.
    SI-RMSE = sqrt(mean(d_i²) - lambda·mean(d_i)²)
    where d_i = log(pred_i) - log(gt_i)
    """
    if mask is None:
        mask = np.ones_like(gt, dtype=bool)
    p = np.log(np.clip(pred[mask], 1e-8, None))
    g = np.log(np.clip(gt[mask],   1e-8, None))
    d = p - g
    lam = 0.5   # Eigen 논문 기준
    return float(np.sqrt(np.mean(d**2) - lam * np.mean(d)**2))


# ── 합성 데이터로 지표 검증 ──
np.random.seed(42)
H_val, W_val = 100, 100
gt_depth   = np.random.uniform(0.5, 10.0, (H_val, W_val))  # GT 깊이 (m)
noise      = np.random.normal(0, 0.5, (H_val, W_val))
pred_depth = gt_depth * 2.3 + 1.0 + noise  # 2.3배 스케일 + 1m 오프셋 + 노이즈

# 정렬 전
rmse_raw    = compute_rmse(pred_depth, gt_depth)
absrel_raw  = compute_abs_rel(pred_depth, gt_depth)
delta1_raw  = compute_delta(pred_depth, gt_depth)
si_raw      = compute_si_rmse(pred_depth, gt_depth)

# 정렬 후 (scale-shift)
pred_aligned, s, t = align_scale_shift(pred_depth, gt_depth)
rmse_align   = compute_rmse(pred_aligned, gt_depth)
absrel_align = compute_abs_rel(pred_aligned, gt_depth)
delta1_align = compute_delta(pred_aligned, gt_depth)
si_align     = compute_si_rmse(pred_aligned, gt_depth)

print('=' * 55)
print(f'{'지표':<20} {'정렬 전':>12} {'정렬 후':>12}')
print('-' * 55)
print(f'{'RMSE (m)':<20} {rmse_raw:>12.4f} {rmse_align:>12.4f}')
print(f'{'AbsRel':<20} {absrel_raw:>12.4f} {absrel_align:>12.4f}')
print(f'{'δ1 (acc ↑)':<20} {delta1_raw:>12.4f} {delta1_align:>12.4f}')
print(f'{'SI-RMSE':<20} {si_raw:>12.4f} {si_align:>12.4f}')
print('=' * 55)
print(f'\n[스케일 정렬] s={s:.3f}, t={t:.3f}')
print('→ 정렬 후 RMSE가 급감: 예측값의 스케일 문제를 보정했기 때문')
print('→ SI-RMSE는 정렬 전후 차이 작음: 스케일에 자체적으로 불변')

In [ ]:
# ── 평가 지표 시각화 ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# GT vs Pred (정렬 전)
gt_flat  = gt_depth.flatten()
pred_flat_raw = pred_depth.flatten()
pred_flat_ali = pred_aligned.flatten()

sample_idx = np.random.choice(len(gt_flat), 500, replace=False)

axes[0].scatter(gt_flat[sample_idx], pred_flat_raw[sample_idx], alpha=0.3, s=10, color='red', label=f'정렬 전 (RMSE={rmse_raw:.2f}m)')
axes[0].scatter(gt_flat[sample_idx], pred_flat_ali[sample_idx], alpha=0.3, s=10, color='blue', label=f'정렬 후 (RMSE={rmse_align:.2f}m)')
lim = max(gt_flat.max(), pred_flat_ali.max())
axes[0].plot([0,lim],[0,lim], 'k--', alpha=0.5, label='완벽한 예측 (y=x)')
axes[0].set_xlabel('GT 깊이 (m)'); axes[0].set_ylabel('예측 깊이 (m)')
axes[0].set_title('Scale 정렬 효과')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# δ1 이해
thresholds = [1.25**1, 1.25**2, 1.25**3]
delta_vals_raw  = [compute_delta(pred_depth, gt_depth, th) for th in thresholds]
delta_vals_ali  = [compute_delta(pred_aligned, gt_depth, th) for th in thresholds]
x_pos = np.arange(3)
axes[1].bar(x_pos - 0.2, delta_vals_raw, 0.4, label='정렬 전', color='red', alpha=0.7)
axes[1].bar(x_pos + 0.2, delta_vals_ali, 0.4, label='정렬 후', color='blue', alpha=0.7)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(['δ1 (1.25)', 'δ2 (1.25²)', 'δ3 (1.25³)'])
axes[1].set_ylabel('비율 (↑높을수록 좋음)')
axes[1].set_ylim(0, 1.1)
axes[1].set_title('δ 정확도 지표 비교')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

# 오차 분포
err_raw = np.abs(pred_flat_raw - gt_flat)
err_ali = np.abs(pred_flat_ali - gt_flat)
axes[2].hist(err_raw, bins=30, alpha=0.6, color='red', label=f'정렬 전 (중앙값={np.median(err_raw):.2f}m)', density=True)
axes[2].hist(err_ali, bins=30, alpha=0.6, color='blue', label=f'정렬 후 (중앙값={np.median(err_ali):.2f}m)', density=True)
axes[2].set_xlabel('절대 오차 (m)'); axes[2].set_ylabel('밀도')
axes[2].set_title('오차 분포 비교')
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

plt.suptitle('Depth Estimation 평가 지표 — Scale Alignment 효과', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. DepthAnythingV2 아키텍처

### 핵심 구성요소

```
입력 이미지 (H×W×3)
        │
        ▼
┌─────────────────────────────────────────┐
│  DINOv2 Encoder (ViT-Large)             │
│  - 자기지도학습(Self-Supervised) 사전학습  │
│  - 14×14 patch → 37×37 feature tokens   │
│  - 풍부한 의미론적 특징 추출              │
└─────────────────────────────────────────┘
        │  (multi-scale features)
        ▼
┌─────────────────────────────────────────┐
│  DPT Decoder (Dense Prediction Transformer) │
│  - ViT 토큰을 픽셀 레벨로 upsampling     │
│  - 4개 스케일의 feature를 fusion         │
│  - 최종: H×W 깊이 맵 출력               │
└─────────────────────────────────────────┘
        │
        ▼
  깊이 맵 (H×W, 상대적 깊이 0~1)
```

### v1 vs v2 차이
| 항목 | DepthAnythingV1 | DepthAnythingV2 |
|------|----------------|----------------|
| 인코더 | ViT (DINOv2) | DINOv2 (동일) |
| 학습 데이터 | 실제 이미지 중심 | **합성 데이터 추가** (개선 핵심) |
| 세밀한 영역 | 약함 | 개선됨 |
| 메트릭 깊이 | 불안정 | **Metric Depth 모델** 별도 제공 |

### 왜 DINOv2를 인코더로 쓰나?
```
DINOv2: 학습 시 깊이 레이블 없음 → 순수 이미지의 구조 학습
→ 객체 경계, 텍스처, 원근감에 자연스럽게 민감한 특징 형성
→ 깊이 추정에 최적의 사전 특징 제공
```

In [ ]:
# ── DepthAnythingV2 모델 로드 (transformers) ──
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from PIL import Image

# Small 모델 (~97MB 자동 다운로드)
MODEL_ID = 'depth-anything/Depth-Anything-V2-Small-hf'
print(f'모델 로드 중: {MODEL_ID}')

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
depth_model = AutoModelForDepthEstimation.from_pretrained(MODEL_ID)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
depth_model = depth_model.to(device).eval()

print(f'\n로드 완료 → device: {device}')
print(f'파라미터 수: {sum(p.numel() for p in depth_model.parameters()):,}')

In [ ]:
def predict_depth(img_path: str) -> tuple:
    """
    단일 이미지 깊이 추정.
    반환: (depth_map: H×W ndarray, img_rgb: H×W×3 ndarray)
    """
    img_pil = Image.open(img_path).convert('RGB')
    inputs  = processor(images=img_pil, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = depth_model(**inputs)
    # 원본 이미지 크기로 보간
    depth = torch.nn.functional.interpolate(
        outputs.predicted_depth.unsqueeze(1),
        size=img_pil.size[::-1],
        mode='bilinear',
        align_corners=False
    ).squeeze().cpu().numpy()
    img_rgb = np.array(img_pil)
    return depth, img_rgb


# 테스트 추론
img_files = sorted(COCO_IMG_DIR.glob('*.jpg'))
test_path  = str(img_files[0])
depth_map, img_rgb = predict_depth(test_path)

print(f'이미지: {Path(test_path).name}')
print(f'깊이 맵 shape: {depth_map.shape}')
print(f'깊이 범위: {depth_map.min():.2f} ~ {depth_map.max():.2f}  (상대 단위)')

# 정규화 (0~1)
depth_norm = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min() + 1e-8)

# ── 시각화 ──
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_rgb)
axes[0].set_title('원본 이미지', fontsize=11)
axes[0].axis('off')

im1 = axes[1].imshow(depth_norm, cmap='plasma')
plt.colorbar(im1, ax=axes[1], fraction=0.046)
axes[1].set_title('깊이 맵 (plasma 컬러맵)\n밝을수록 가까움', fontsize=11)
axes[1].axis('off')

# 역깊이 (먼 거리=밝음 표현)
im2 = axes[2].imshow(1 - depth_norm, cmap='gray')
plt.colorbar(im2, ax=axes[2], fraction=0.046)
axes[2].set_title('역깊이 맵 (gray)\n밝을수록 멀리 있음', fontsize=11)
axes[2].axis('off')

plt.suptitle('DepthAnythingV2 — 단일 이미지 추론', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 여러 이미지 깊이 맵 비교 (2×3 그리드) ──
n_show = 6
sample_imgs = img_files[:n_show]

fig, axes = plt.subplots(2, n_show, figsize=(4*n_show, 8))

cmaps = ['plasma', 'magma', 'inferno', 'viridis', 'cividis', 'turbo']

for col, img_f in enumerate(sample_imgs):
    dm, img_r = predict_depth(str(img_f))
    dm_norm = (dm - dm.min()) / (dm.max() - dm.min() + 1e-8)

    axes[0][col].imshow(img_r)
    axes[0][col].set_title(img_f.name[:16], fontsize=7)
    axes[0][col].axis('off')

    axes[1][col].imshow(dm_norm, cmap='plasma')
    axes[1][col].set_title(f'min={dm.min():.1f} max={dm.max():.1f}', fontsize=7)
    axes[1][col].axis('off')

axes[0][0].set_ylabel('원본', fontsize=10)
axes[1][0].set_ylabel('깊이 맵', fontsize=10)
plt.suptitle('COCO128 다중 이미지 깊이 추정 결과', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n[깊이 맵 해석 주의사항]')
print('- 출력값은 상대 깊이 (절대 미터 단위 아님)')
print('- 값이 클수록 가까움 (모델에 따라 반대일 수 있음)')
print('- 절대 미터 변환은 카메라 캘리브레이션 + 스케일 추정 필요')

## 3. Scale-Invariant 평가 파이프라인

### 전략
COCO128에는 GT 깊이 맵이 없으므로:
1. **합성 깊이 GT** — 이미지 픽셀 밝기를 거리로 변환 (원근감 근사)
2. **Scale 정렬 후 평가** — 예측값과 GT를 선형 정렬하여 RMSE/δ1 계산
3. **상대적 성능 비교** — 이미지마다 깊이 분포 특성 분석

> 실제 연구에서는 NYU Depth v2 (실내) 또는 KITTI (자율주행 야외) GT 깊이 사용.
> Phase 4 CARLA에서 시뮬레이터 Depth 센서로 완전한 GT 평가 가능.

In [ ]:
def synthetic_gt_depth(img_rgb: np.ndarray) -> np.ndarray:
    """
    합성 GT 깊이: 이미지 밝기 + 수직 위치 기반 원근감 근사.
    (어두울수록 멀리, 이미지 하단일수록 가까움 — 야외 원근감 휴리스틱)
    """
    H, W = img_rgb.shape[:2]
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    # 수직 위치 가중치 (하단=1.0 가까움, 상단=0.1 멈)
    row_weight = np.linspace(0.1, 1.0, H)[:, np.newaxis] * np.ones((1, W))
    # 깊이 = 밝기의 역수 × 수직 위치
    depth_gt = (1.0 - gray * 0.5) * row_weight
    depth_gt = np.clip(depth_gt, 0.1, 10.0)  # 0.1~10m 범위
    return depth_gt


# ── 다중 이미지 평가 ──
print('Scale-Invariant 평가 진행 중...')
eval_results = []

for img_f in img_files[:20]:
    dm, img_r = predict_depth(str(img_f))
    gt = synthetic_gt_depth(img_r)

    # Scale 정렬
    dm_aligned, s, t = align_scale_shift(dm, gt)
    valid = (gt > 0.1) & (gt < 9.9)

    rmse   = compute_rmse(dm_aligned, gt, valid)
    absrel = compute_abs_rel(dm_aligned, gt, valid)
    delta1 = compute_delta(dm_aligned, gt, 1.25, valid)
    delta2 = compute_delta(dm_aligned, gt, 1.25**2, valid)
    si     = compute_si_rmse(dm, gt, valid)  # SI-RMSE는 정렬 없이도 유효

    eval_results.append({
        'name': img_f.name,
        'rmse': rmse, 'absrel': absrel,
        'delta1': delta1, 'delta2': delta2, 'si_rmse': si
    })

print(f'\n평가 완료: {len(eval_results)}장')
print()
print(f'{'지표':<15} {'평균':>10} {'중앙값':>10} {'표준편차':>10}')
print('-' * 50)
for key, name in [('rmse','RMSE(m)↓'), ('absrel','AbsRel↓'),
                   ('delta1','δ1(acc)↑'), ('delta2','δ2(acc)↑'), ('si_rmse','SI-RMSE↓')]:
    vals = [r[key] for r in eval_results]
    print(f'{name:<15} {np.mean(vals):>10.4f} {np.median(vals):>10.4f} {np.std(vals):>10.4f}')

In [ ]:
# ── 평가 결과 시각화 ──
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

metrics = ['rmse', 'absrel', 'si_rmse', 'delta1', 'delta2']
metric_labels = ['RMSE↓', 'AbsRel↓', 'SI-RMSE↓', 'δ1 (acc)↑', 'δ2 (acc)↑']
colors_m = ['#E74C3C', '#F39C12', '#8E44AD', '#27AE60', '#2980B9']

for i, (m, label, col) in enumerate(zip(metrics, metric_labels, colors_m)):
    ax = axes[i // 3][i % 3]
    vals = [r[m] for r in eval_results]
    ax.bar(range(len(vals)), vals, color=col, alpha=0.8)
    ax.axhline(y=np.mean(vals), color="black", linestyle="--",
               label=f"평균={np.mean(vals):.4f}")
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('이미지 인덱스')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

# 마지막 칸: 지표 요약 바 차트
ax_sum = axes[1][2]
summary_names = ['RMSE(낮을수록↓)', 'AbsRel(낮을수록↓)', 'SI-RMSE(낮을수록↓)', 'δ1 acc(높을수록↑)', 'δ2 acc(높을수록↑)']
summary_vals_n = [
    1 - min(np.mean([r['rmse']    for r in eval_results]) / 2.0, 1.0),
    1 - min(np.mean([r['absrel']  for r in eval_results]) / 1.0, 1.0),
    1 - min(np.mean([r['si_rmse'] for r in eval_results]) / 1.0, 1.0),
    np.mean([r['delta1'] for r in eval_results]),
    np.mean([r['delta2'] for r in eval_results]),
]
bar_c = ['#E74C3C','#F39C12','#8E44AD','#27AE60','#2980B9']
ax_sum.barh(summary_names, summary_vals_n, color=bar_c, alpha=0.8)
ax_sum.set_xlim(0, 1)
ax_sum.set_title('정규화 성능 요약', fontsize=11)
ax_sum.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
ax_sum.grid(True, alpha=0.3, axis='x')

plt.suptitle('DepthAnythingV2 Scale-Invariant 평가 결과', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. YOLOv8 Detection + Depth 연동 — 객체별 거리 추정

### 파이프라인
```
이미지
  ├─→ YOLOv8 → bbox [x1,y1,x2,y2] + class
  └─→ DepthAnythingV2 → 깊이 맵
          ↓
  bbox 내부 깊이 맵의 중앙값 → 객체까지 상대 거리
          ↓
  거리 기반 위험도 분류
```

### 왜 중앙값(Median)을 쓰나?
```
평균: 객체 뒤 배경 픽셀이 섞이면 편향 발생
중앙값: 배경/이상치에 강건 → 객체 본체의 깊이에 가까움
```

In [ ]:
from ultralytics import YOLO

det_model = YOLO(PROJECT_ROOT / 'phase1_basics/detection/runs/detect/coco128_finetune/weights/best.pt')
print('YOLOv8 파인튜닝 모델 로드 완료')

COCO_NAMES = [
    'person','bicycle','car','motorcycle','airplane','bus','train','truck',
    'boat','traffic light','fire hydrant','stop sign','parking meter','bench',
    'bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe',
    'backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard',
    'sports ball','kite','baseball bat','baseball glove','skateboard','surfboard',
    'tennis racket','bottle','wine glass','cup','fork','knife','spoon','bowl',
    'banana','apple','sandwich','orange','broccoli','carrot','hot dog','pizza',
    'donut','cake','chair','couch','potted plant','bed','dining table','toilet',
    'tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven',
    'toaster','sink','refrigerator','book','clock','vase','scissors','teddy bear',
    'hair drier','toothbrush'
]
AV_CLASSES_ID = {0,1,2,3,5,7,9,11}  # person, bicycle, car, motorcycle, bus, truck, traffic light, stop sign

# 위험도 분류 (상대 깊이 기준)
def classify_risk(depth_val: float, d_min: float, d_max: float) -> tuple:
    """
    상대 깊이를 0~1 정규화 후 위험도 분류.
    depth가 클수록 가까움 (DepthAnythingV2 기준).
    """
    norm = (depth_val - d_min) / (d_max - d_min + 1e-8)
    if norm > 0.75:
        return '위험', (220, 50, 50)
    elif norm > 0.45:
        return '주의', (220, 150, 50)
    else:
        return '안전', (50, 180, 50)


# ── 추론 ──
# 객체가 많은 이미지 선택
best_img = None
best_count = 0
for img_f in img_files[:30]:
    res = det_model(str(img_f), verbose=False, conf=0.25)
    if res[0].boxes is not None and len(res[0].boxes) > best_count:
        best_count = len(res[0].boxes)
        best_img = img_f
    if best_count >= 4:
        break

print(f'선택 이미지: {best_img.name}  ({best_count}개 객체 검출)')
det_results = det_model(str(best_img), verbose=False, conf=0.25)
depth_map_full, img_rgb_full = predict_depth(str(best_img))

In [ ]:
# ── Detection + Depth 통합 시각화 ──
boxes_det  = det_results[0].boxes
d_min_full = depth_map_full.min()
d_max_full = depth_map_full.max()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 원본 + Detection
vis_det = img_rgb_full.copy()
depth_norm_full = (depth_map_full - d_min_full) / (d_max_full - d_min_full + 1e-8)

obj_info = []
if boxes_det is not None:
    for box in boxes_det:
        x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
        cls_id = int(box.cls[0])
        conf   = float(box.conf[0])
        cls_name = COCO_NAMES[cls_id] if cls_id < len(COCO_NAMES) else str(cls_id)

        # bbox 내부 중앙 영역 깊이 중앙값
        cy_box = (y1+y2)//2; cx_box = (x1+x2)//2
        h_box = max(1, (y2-y1)//4); w_box = max(1, (x2-x1)//4)
        roi = depth_map_full[
            max(0,cy_box-h_box):min(depth_map_full.shape[0],cy_box+h_box),
            max(0,cx_box-w_box):min(depth_map_full.shape[1],cx_box+w_box)
        ]
        depth_val = float(np.median(roi)) if roi.size > 0 else 0.0
        risk, rgb_c = classify_risk(depth_val, d_min_full, d_max_full)

        obj_info.append({'cls': cls_name, 'cls_id': cls_id, 'conf': conf,
                         'box': (x1,y1,x2,y2), 'depth': depth_val, 'risk': risk, 'color': rgb_c})

        # Detection 시각화
        cv2.rectangle(vis_det, (x1,y1), (x2,y2), rgb_c, 2)
        label_text = f'{cls_name} {conf:.2f}'
        cv2.putText(vis_det, label_text, (x1, max(y1-5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, rgb_c, 1)

axes[0].imshow(vis_det)
axes[0].set_title(f'YOLOv8 Detection ({len(obj_info)}개)', fontsize=11)
axes[0].axis('off')

# 깊이 맵
im = axes[1].imshow(depth_norm_full, cmap='plasma')
plt.colorbar(im, ax=axes[1], fraction=0.046)
for info in obj_info:
    x1,y1,x2,y2 = info['box']
    rect = plt.Rectangle((x1,y1), x2-x1, y2-y1, fill=False,
                          edgecolor=[c/255 for c in info['color']], linewidth=2)
    axes[1].add_patch(rect)
axes[1].set_title('깊이 맵 + Detection bbox', fontsize=11)
axes[1].axis('off')

# 위험도 통합 시각화
vis_risk = img_rgb_full.copy().astype(np.float32)
# 깊이 맵을 반투명 오버레이
plasma = plt.get_cmap('plasma')
depth_colored = (plasma(depth_norm_full)[:,:,:3] * 255).astype(np.float32)
vis_risk = vis_risk * 0.6 + depth_colored * 0.4

for info in obj_info:
    x1,y1,x2,y2 = info['box']
    color = info['color']
    cv2.rectangle(vis_risk.astype(np.uint8), (x1,y1), (x2,y2), color, 3)
    risk_label = f"{info['cls']} [{info['risk']}]"
    cv2.putText(vis_risk.astype(np.uint8), risk_label, (x1, max(y1-8,12)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

axes[2].imshow(vis_risk.astype(np.uint8))
axes[2].set_title('위험도 분류 (빨강=위험, 주황=주의, 초록=안전)', fontsize=11)
axes[2].axis('off')
legend_patches = [
    mpatches.Patch(color=[220/255,50/255,50/255], label='위험 (근거리)'),
    mpatches.Patch(color=[220/255,150/255,50/255], label='주의 (중거리)'),
    mpatches.Patch(color=[50/255,180/255,50/255], label='안전 (원거리)'),
]
axes[2].legend(handles=legend_patches, loc='lower left', fontsize=8)

plt.suptitle('YOLOv8 + DepthAnythingV2 통합 — 객체 거리 기반 위험도', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n[객체별 상세 정보]')
print(f'{"클래스":<15} {"신뢰도":>8} {"깊이(상대)": >12} {"위험도":>8}')
print('-' * 48)
for info in sorted(obj_info, key=lambda x: -x['depth']):
    print(f"{info['cls']:<15} {info['conf']:>8.3f} {info['depth']:>12.2f} {info['risk']:>8}")

In [ ]:
# ── 자율주행 핵심 클래스 거리 분포 분석 ──
# 여러 이미지에서 AV 클래스 객체의 상대 깊이 수집
print('자율주행 핵심 클래스 깊이 분포 수집 중...')

from collections import defaultdict
cls_depths = defaultdict(list)

for img_f in img_files[:20]:
    det_r = det_model(str(img_f), verbose=False, conf=0.25)
    if det_r[0].boxes is None:
        continue
    dm, _ = predict_depth(str(img_f))
    dm_n = (dm - dm.min()) / (dm.max() - dm.min() + 1e-8)
    for box in det_r[0].boxes:
        cls_id = int(box.cls[0])
        if cls_id not in AV_CLASSES_ID:
            continue
        x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
        cy_b=(y1+y2)//2; cx_b=(x1+x2)//2
        h_b=max(1,(y2-y1)//4); w_b=max(1,(x2-x1)//4)
        roi = dm_n[max(0,cy_b-h_b):min(dm_n.shape[0],cy_b+h_b),
                   max(0,cx_b-w_b):min(dm_n.shape[1],cx_b+w_b)]
        if roi.size > 0:
            cls_name = COCO_NAMES[cls_id]
            cls_depths[cls_name].append(float(np.median(roi)))

print(f'수집 완료: {sum(len(v) for v in cls_depths.values())}개 객체')

if cls_depths:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 박스플롯
    cls_names_sorted = sorted(cls_depths.keys(), key=lambda c: -np.mean(cls_depths[c]))
    data = [cls_depths[c] for c in cls_names_sorted]
    bp = axes[0].boxplot(data, labels=cls_names_sorted, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('#AED6F1')
    axes[0].set_ylabel('상대 깊이 (클수록 가까움)')
    axes[0].set_title('자율주행 클래스별 깊이 분포')
    axes[0].grid(True, alpha=0.3, axis='y')
    axes[0].tick_params(axis='x', rotation=30)

    # 평균 깊이 바 차트 + 위험도 색상
    mean_depths = [np.mean(cls_depths[c]) for c in cls_names_sorted]
    risk_colors = ['#E74C3C' if d > 0.65 else '#F39C12' if d > 0.4 else '#27AE60' for d in mean_depths]
    axes[1].bar(cls_names_sorted, mean_depths, color=risk_colors, alpha=0.85)
    axes[1].axhline(y=0.65, color='red', linestyle='--', alpha=0.7, label='위험 임계값')
    axes[1].axhline(y=0.40, color='orange', linestyle='--', alpha=0.7, label='주의 임계값')
    axes[1].set_ylabel('평균 상대 깊이')
    axes[1].set_title('클래스별 평균 거리 (위험도 색상)')
    axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')
    axes[1].tick_params(axis='x', rotation=30)

    plt.suptitle('자율주행 핵심 클래스 거리 분포 분석', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('[자율주행 클래스 객체가 현재 이미지에 없습니다]')

## 5. 한계 및 Phase 4 CARLA 연결

### 현재 파이프라인의 한계
```
1. 절대 거리(m) 미지원
   - 상대 깊이만 출력 → 실제 m 변환 불가
   - CARLA에서 카메라 캘리브레이션 + 깊이 센서 GT 제공 → 해결 가능

2. 합성 GT 사용
   - 실제 GT 깊이 맵(NYU, KITTI) 없음 → 평가의 절대적 신뢰도 낮음
   - CARLA 시뮬레이터: depth camera sensor → 완전한 GT 자동 생성

3. 실시간 성능 미측정
   - DepthAnythingV2-Small: RTX 4080에서 약 30-50fps 예상
   - Phase 4에서 실측 필요
```

### Segmentation + Depth 시너지 (Phase 4)
```
SAM2 마스크 + Depth 맵
    ↓
마스크 영역 내 깊이 중앙값 → 객체의 3D 부피 추정
    ↓
"보행자까지 3.2m, 차량까지 8.7m" → 자율주행 경로 계획 입력
```

### Phase 1 → Phase 4 연결 고리
| Phase 1 | Phase 4 CARLA |
|---------|---------------|
| YOLOv8 Detection | CARLA 데이터로 재평가 |
| SAM2 Segmentation | CARLA semantic GT와 mIoU 측정 |
| DepthAnythingV2 | CARLA depth sensor GT와 RMSE 측정 |
| ByteTrack Tracking | CARLA 동적 씬 추적 평가 |
| BEV IPM | CARLA 카메라 캘리브레이션 활용 |

## 6. 최종 결과 요약 및 Phase 4 연결

In [ ]:
print('=' * 65)
print('Phase 1 Depth Estimation 파이프라인 최종 결과')
print('=' * 65)
print()
print('[구현 완료]')
print('  1. RMSE         — 깊이 오차 제곱 평균 직접 구현')
print('  2. AbsRel       — 상대 오차 (스케일 무관) 직접 구현')
print('  3. δ1 / δ2      — 픽셀 정확도 지표 직접 구현')
print('  4. SI-RMSE      — Scale-Invariant 지표 (Eigen 2014) 직접 구현')
print('  5. 스케일 정렬   — 최소제곱법 align_scale_shift 구현')
print('  6. DepthAnythingV2 추론 (transformers, GPU 가속)')
print('  7. Detection + Depth 통합 위험도 분류 시스템')
print()
print('[평가 결과] (Scale 정렬 후, 합성 GT 기준)')
if eval_results:
    print(f'  RMSE         : {np.mean([r["rmse"]   for r in eval_results]):.4f} m')
    print(f'  AbsRel       : {np.mean([r["absrel"] for r in eval_results]):.4f}')
    print(f'  δ1 정확도    : {np.mean([r["delta1"] for r in eval_results]):.4f}')
    print(f'  SI-RMSE      : {np.mean([r["si_rmse"] for r in eval_results]):.4f}')
print()
print('[Phase 1 완성 요약]')
print('  Detection  — YOLOv8 + mAP 파이프라인              ✅')
print('  Segmentation — SAM2 + mIoU 파이프라인             ✅')
print('  Depth      — DepthAnythingV2 + RMSE/δ1 파이프라인 ✅')
print()
print('[Phase 4 CARLA 연결]')
print('  → CARLA depth sensor: 절대 거리 GT 자동 생성')
print('  → CARLA semantic: mIoU 절대 평가 가능')
print('  → 현재 파이프라인 그대로 CARLA 데이터에 적용')
print('=' * 65)